# E1: Typed Failure Telemetry vs 1-bit Feedback (miniF2F, Local Qwen2.5-Coder-7B)

Runs the pre-registered E1 experiment (`docs/experiments/e1_preregistration.md`):
three feedback arms (`a0_onebit`, `a1_raw_error`, `a2_typed_packet`) under an
identical model, instruction text, and per-theorem call budget, audited against Lean.

Decision rule (fixed before execution): the paired bootstrap 95% CI of
`solve_rate(a2) - solve_rate(a1)` must exclude zero in favor of a2.

Order of cells: setup -> start Colab local LLM server -> endpoint smoke -> dry smoke (no API cost) -> Lean toolchain -> local LLM run -> report.

In [ ]:
# 1. Clone the repo and install
REPO_URL = "https://github.com/abhorrence-of-Gods/lean-rgc-automation-stack.git"
import os
if not os.path.exists("lean-rgc-automation-stack"):
    !git clone --depth 1 {REPO_URL}
%cd lean-rgc-automation-stack
!pip install -q -e .
!lean-rgc --help | head -5

In [ ]:
# 2. Optional: mount Drive so the audit cache and LLM cache survive session resets
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/lean_rgc_e1"
else:
    PERSIST = "/content/lean_rgc_e1"
WORK_ROOT = "/content/lean_rgc_e1_work"
MINIF2F_REPO = f"{WORK_ROOT}/minif2f"
os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(f"{PERSIST}/llm_cache", exist_ok=True)
os.makedirs(f"{PERSIST}/runs", exist_ok=True)
AUDIT_CACHE_DB = f"{PERSIST}/audit_cache.sqlite"
print("persist dir:", PERSIST)
print("work dir:", WORK_ROOT)

In [ ]:
# 3. Local LLM endpoint (Qwen2.5-Coder-7B-Instruct; OpenAI-compatible)
# If this notebook runs in a hosted Colab runtime, point this to a tunnel URL
# or use a Colab local runtime so 127.0.0.1 reaches your model server.
import os
LOCAL_LLM_MODEL = os.environ.get("LEAN_RGC_LOCAL_LLM_MODEL", "Qwen/Qwen2.5-Coder-7B-Instruct")
LOCAL_LLM_BASE_URL = os.environ.get("LEAN_RGC_LOCAL_LLM_BASE_URL", "http://127.0.0.1:8000/v1")
LOCAL_LLM_JSON_MODE = os.environ.get("LEAN_RGC_LOCAL_LLM_JSON_MODE", "0").strip().lower() in {"1", "true", "yes"}
LOCAL_LLM_API_KEY = os.environ.get("LEAN_RGC_LLM_API_KEY", "").strip()
if not LOCAL_LLM_API_KEY:
    os.environ.pop("LEAN_RGC_LLM_API_KEY", None)
print("local llm model:", LOCAL_LLM_MODEL)
print("local llm base_url:", LOCAL_LLM_BASE_URL)
print("local llm api key:", "set" if LOCAL_LLM_API_KEY else "not set")
print("json response_format:", LOCAL_LLM_JSON_MODE)

In [ ]:
# 3b. Start a local OpenAI-compatible server in Colab for Qwen2.5-Coder-7B
# Default backend is transformers+bitsandbytes because plain pip vLLM can pick a CUDA wheel
# that is incompatible with Colab. Set LEAN_RGC_COLAB_LLM_BACKEND=vllm to try vLLM.
import os, socket, subprocess, sys, textwrap, time, urllib.request

os.environ.setdefault("HF_HOME", f"{PERSIST}/hf_cache")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

def truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes", "on"}

start_setting = os.environ.get("LEAN_RGC_START_COLAB_LOCAL_LLM", "auto").strip().lower()
START_COLAB_LOCAL_LLM = LOCAL_LLM_BASE_URL.startswith("http://127.0.0.1:") if start_setting == "auto" else truthy(start_setting)
COLAB_LLM_BACKEND = os.environ.get("LEAN_RGC_COLAB_LLM_BACKEND", "transformers").strip().lower()
SERVER_LOG = f"{WORK_ROOT}/qwen25_coder_7b_{COLAB_LLM_BACKEND}.log"
SERVER_SCRIPT = f"{WORK_ROOT}/qwen25_openai_server.py"

def port_open(host="127.0.0.1", port=8000, timeout=1.0):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0

if START_COLAB_LOCAL_LLM:
    if port_open():
        print("local LLM server already has port 8000 open")
    else:
        if COLAB_LLM_BACKEND == "vllm":
            print("installing vLLM with uv auto torch backend...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
            subprocess.check_call(["uv", "pip", "install", "--system", "-q", "vllm", "--torch-backend=auto"])
            cmd = [
                "vllm", "serve", LOCAL_LLM_MODEL,
                "--served-model-name", LOCAL_LLM_MODEL,
                "--host", "0.0.0.0",
                "--port", "8000",
                "--dtype", "auto",
                "--gpu-memory-utilization", "0.85",
                "--max-model-len", "8192",
                "--trust-remote-code",
            ]
        else:
            print("installing Transformers local server dependencies...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.45", "accelerate", "bitsandbytes", "fastapi", "uvicorn",
            ])
            server_code = r'''
import os
import time
from threading import Lock
from typing import Any

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from fastapi import FastAPI
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("LEAN_RGC_LOCAL_LLM_MODEL", "Qwen/Qwen2.5-Coder-7B-Instruct")
QUANT = os.environ.get("LEAN_RGC_TRANSFORMERS_QUANT", "4bit").strip().lower()
MAX_MODEL_LEN = int(os.environ.get("LEAN_RGC_LOCAL_MAX_MODEL_LEN", "8192"))

print(f"loading tokenizer: {MODEL_ID}", flush=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs: dict[str, Any] = {"device_map": "auto", "trust_remote_code": True}
if torch.cuda.is_available() and QUANT == "4bit":
    try:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        print("loading model in 4bit", flush=True)
    except Exception as exc:
        print(f"4bit unavailable, falling back to fp16: {exc}", flush=True)
        model_kwargs["torch_dtype"] = torch.float16
elif torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float32

print(f"loading model: {MODEL_ID}", flush=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
first_device = next((p.device for p in model.parameters() if p.device.type != "meta"), torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(f"server ready on device {first_device}", flush=True)

app = FastAPI()
lock = Lock()

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "local"}]}

@app.post("/v1/chat/completions")
def chat_completions(payload: dict[str, Any]):
    messages = payload.get("messages") or []
    max_tokens = int(payload.get("max_tokens") or 1024)
    max_tokens = max(1, min(max_tokens, 2048))
    temperature = float(payload.get("temperature") or 0.0)
    top_p = float(payload.get("top_p") or 1.0)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_MODEL_LEN)
    encoded = {k: v.to(first_device) for k, v in encoded.items()}
    gen_kwargs: dict[str, Any] = {
        "max_new_tokens": max_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        gen_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": top_p})
    else:
        gen_kwargs["do_sample"] = False
    with lock:
        with torch.inference_mode():
            output = model.generate(**encoded, **gen_kwargs)
    input_tokens = int(encoded["input_ids"].shape[-1])
    completion_ids = output[0][input_tokens:]
    text = tokenizer.decode(completion_ids, skip_special_tokens=True)
    now = int(time.time())
    return {
        "id": f"chatcmpl-local-{now}",
        "object": "chat.completion",
        "created": now,
        "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {
            "prompt_tokens": input_tokens,
            "completion_tokens": int(completion_ids.shape[-1]),
            "total_tokens": int(output.shape[-1]),
        },
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
'''
            with open(SERVER_SCRIPT, "w", encoding="utf-8") as f:
                f.write(server_code)
            cmd = [sys.executable, SERVER_SCRIPT]

        print("starting local LLM server; backend:", COLAB_LLM_BACKEND)
        print("server log:", SERVER_LOG)
        log = open(SERVER_LOG, "ab", buffering=0)
        env = os.environ.copy()
        env["LEAN_RGC_LOCAL_LLM_MODEL"] = LOCAL_LLM_MODEL
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
        print("local LLM server pid:", proc.pid)

    models_url = LOCAL_LLM_BASE_URL.rstrip("/") + "/models"
    for attempt in range(180):
        try:
            with urllib.request.urlopen(models_url, timeout=2) as resp:
                print("local LLM server ready:", resp.read().decode("utf-8")[:500])
                break
        except Exception:
            if attempt % 6 == 0:
                print(f"waiting for local LLM server... {attempt * 10}s")
            time.sleep(10)
    else:
        if os.path.exists(SERVER_LOG):
            print(open(SERVER_LOG, "r", encoding="utf-8", errors="replace").read()[-6000:])
        raise RuntimeError("local LLM server did not become ready; inspect the log above")
else:
    print("skipping Colab local LLM startup; using configured endpoint:", LOCAL_LLM_BASE_URL)


In [ ]:
# 4. LLM config (local Qwen2.5-Coder-7B via OpenAI-compatible chat completions)
import json
LLM_CONFIG = {
    "provider": "openai_compatible",
    "model": LOCAL_LLM_MODEL,
    "base_url": LOCAL_LLM_BASE_URL,
    "api_key_env": "LEAN_RGC_LLM_API_KEY",
    "temperature": 0.2,
    "top_p": 0.95,
    "max_tokens": 1024,
    "seed": 0,
    "timeout_s": 240.0,
    "cache_dir": f"{PERSIST}/llm_cache",
}
if LOCAL_LLM_JSON_MODE:
    LLM_CONFIG["response_format"] = {"type": "json_object"}
with open("llm.json", "w") as f:
    json.dump(LLM_CONFIG, f, indent=2)
MOCK_CONFIG = {**LLM_CONFIG, "provider": "mock", "mock_responses": [
    json.dumps({"proposals": [{"proposal_kind": "tactic", "lean_tactic": "simp"}]})
]}
with open("llm_mock.json", "w") as f:
    json.dump(MOCK_CONFIG, f, indent=2)
print(json.dumps(LLM_CONFIG, indent=2))

In [ ]:
# 4b. Local LLM endpoint smoke (one tiny chat-completions call, no Lean)
import json, urllib.error, urllib.request

def post_json(url, payload, headers=None, timeout=60):
    body = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    for key, value in (headers or {}).items():
        req.add_header(key, value)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

headers = {}
if os.environ.get("LEAN_RGC_LLM_API_KEY"):
    headers["Authorization"] = "Bearer " + os.environ["LEAN_RGC_LLM_API_KEY"]

base = LOCAL_LLM_BASE_URL.rstrip("/")
payload = {
    "model": LOCAL_LLM_MODEL,
    "messages": [
        {"role": "system", "content": "Return only valid JSON."},
        {"role": "user", "content": "Return {\"ok\": true}."},
    ],
    "temperature": 0,
    "max_tokens": 32,
}

try:
    obj = post_json(base + "/chat/completions", payload, headers=headers)
except urllib.error.HTTPError as exc:
    body = exc.read().decode("utf-8", errors="replace")[:1000]
    raise RuntimeError(f"Local LLM endpoint returned HTTP {exc.code}. Check LOCAL_LLM_MODEL and server logs. Body: {body}") from exc
except urllib.error.URLError as exc:
    raise RuntimeError(
        "Local LLM endpoint is not reachable from this notebook runtime. "
        "If this is hosted Colab, 127.0.0.1 points to the Colab VM, not your PC. "
        "Use Colab local runtime, run the OpenAI-compatible server inside this runtime, "
        "or set LEAN_RGC_LOCAL_LLM_BASE_URL to a tunnel/server URL. "
        f"Current base_url={LOCAL_LLM_BASE_URL!r}."
    ) from exc

choice = (obj.get("choices") or [{}])[0]
text = ((choice.get("message") or {}).get("content") or "")[:300]
print("local llm endpoint smoke: ok")
print("model_version:", obj.get("model", LOCAL_LLM_MODEL))
print("response head:", text)

In [ ]:
# 5. Plumbing smoke (zero API cost, no Lean): mock LLM + dry-run audit on 2 tiny tasks
smoke_tasks = [
    {"task_id": "smoke0", "statement": "True", "imports": []},
    {"task_id": "smoke1", "statement": "1 = 1", "imports": []},
]
with open("smoke_tasks.jsonl", "w") as f:
    f.write("".join(json.dumps(t) + "\n" for t in smoke_tasks))
!lean-rgc eval-run --tasks smoke_tasks.jsonl --arm a1_raw_error \
  --llm-config llm_mock.json --out-dir {PERSIST}/runs/smoke_a1 --run-id smoke_a1 \
  --budget-calls 2 --dry-run --lean-cmd "echo lean" | tail -20

In [ ]:
# 6. Lean toolchain + miniF2F (long: toolchain ~5min, mathlib cache download ~10-20min)
!curl -sSfL https://github.com/leanprover/elan/releases/latest/download/elan-x86_64-unknown-linux-gnu.tar.gz | tar xz
!./elan-init -y --default-toolchain none
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
!lean-rgc minif2f-fetch --out {MINIF2F_REPO} --summary-out {PERSIST}/minif2f_fetch.json
%cd {MINIF2F_REPO}
!~/.elan/bin/lake exe cache get
!~/.elan/bin/lake build
%cd /content/lean-rgc-automation-stack

In [ ]:
# 7. Build the pre-registered task subset (100 miniF2F-test theorems, seed 0)
!lean-rgc minif2f-tasks --repo {MINIF2F_REPO} --split test \
  --out {PERSIST}/minif2f_test_tasks.jsonl --summary-out {PERSIST}/minif2f_tasks_summary.json
!lean-rgc eval-subset --tasks {PERSIST}/minif2f_test_tasks.jsonl \
  --out {PERSIST}/e1_tasks.jsonl --n 100 --seed 0

In [ ]:
# 8. E1: run the three arms (identical budget; audit cache shared across arms per lane rules)
LEAN_CMD = "~/.elan/bin/lake env lean"
WORKDIR = MINIF2F_REPO
for arm in ["a0_onebit", "a1_raw_error", "a2_typed_packet"]:
    print("=== arm:", arm, "===")
    !lean-rgc eval-run --tasks {PERSIST}/e1_tasks.jsonl --arm {arm} \
      --llm-config llm.json --out-dir {PERSIST}/runs/e1_{arm} --run-id e1_{arm} \
      --budget-calls 8 --max-proposals 4 \
      --lean-cmd "{LEAN_CMD}" --workdir {WORKDIR} --import-mode preserve \
      --queue-backend bulk --workers 4 --job-timeout-s 180 \
      --audit-cache-db {AUDIT_CACHE_DB} | tail -15

In [ ]:
# 9. Pre-registered report: paired bootstrap over the identical task set
!lean-rgc eval-report \
  --episodes a0_onebit={PERSIST}/runs/e1_a0_onebit/episodes.jsonl \
             a1_raw_error={PERSIST}/runs/e1_a1_raw_error/episodes.jsonl \
             a2_typed_packet={PERSIST}/runs/e1_a2_typed_packet/episodes.jsonl \
  --out {PERSIST}/runs/e1_report.json --n-bootstrap 10000 --seed 0
report = json.load(open(f"{PERSIST}/runs/e1_report.json"))
primary = report.get("primary_comparison")
if primary:
    verdict = "SUPPORTED" if primary["ci_low"] > 0 else "inconclusive"
    print(f"PRIMARY {primary['arm_a']} - {primary['arm_b']}: delta={primary['mean_delta']:+.3f} "
          f"[{primary['ci_low']:+.3f}, {primary['ci_high']:+.3f}] {verdict}")
for comp in report["paired_comparisons"]:
    verdict = "CI EXCLUDES ZERO" if comp["ci_excludes_zero"] else "inconclusive"
    print(f"{comp['arm_a']} - {comp['arm_b']}: delta={comp['mean_delta']:+.3f} "
          f"[{comp['ci_low']:+.3f}, {comp['ci_high']:+.3f}] {verdict}")

In [ ]:
# 10. Episode store summary (boundary cache rate, per-arm solve rates)
from lean_rgc.pbct.episode_store import summarize_prompt_db
for arm in ["a0_onebit", "a1_raw_error", "a2_typed_packet"]:
    db = f"{PERSIST}/runs/e1_{arm}/prompt.sqlite"
    if os.path.exists(db):
        print(arm, json.dumps(summarize_prompt_db(db)["episodes_by_arm"], indent=2))